### Import Packages

In [ ]:
import io
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import statsmodels.api as sm
from PIL import Image
from scipy.stats import chi2_contingency, kruskal
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from xgboost import XGBClassifier

### Data Preparation

In [ ]:
"""
00_data_prep.py
----------------
Project : Determinants of Financial
          Worries in SAARC Countries
Purpose : Load the FULL Global Findex-style SAARC microdata extract
          (200 raw survey columns), recode the multinomial dependent
          variable (fin45), standardise missing-value tokens, and select
          an expanded, leakage-free predictor set for the analytic file
          used by every downstream script.
"""
DATA_PATH = Path("data")
DATA_PATH.mkdir(exist_ok=True)
RAW_PATH = Path("data/saarc_data_full.csv")
OUT_PATH = Path("data/saarc_clean.csv")
DV = "fin45_cat"

df = pd.read_csv(RAW_PATH)


target_map = {
    "For monthly expenses, such as food, housing, or bills": "Daily/Monthly Expenses",
    "For medical costs in case of a serious illness or accident": "Medical Emergency",
    "For school or education fees": "Education",
    "For their old age": "Old Age",
    "For their business": "Business",
    "Some other reason": "Other/Unspecified",
    "Don't know": "Other/Unspecified",
    "Refused": "Other/Unspecified",
}
df["fin45_cat"] = df["fin45"].map(target_map)
assert df["fin45_cat"].isna().sum() == 0, "Unmapped category found in fin45"

# Standardise missing tokens on the original 15-variable core set
df["fin24a"] = df["fin24a"].fillna("Not Applicable")
df["domestic_remittances"] = df["domestic_remittances"].fillna("Not Applicable")

dkref_tokens = ["DK/ref", "Don't know", "Refused"]
for col in ["pay_utilities", "domestic_remittances", "fin24a", "fin24"]:
    df[col] = df[col].replace(dkref_tokens, "Don't know/Refused")

df["pay_utilities"] = df["pay_utilities"].replace(
    {"Don't know/Refused": "Other/DK", "In some other way": "Other/DK"}
)
df["domestic_remittances"] = df["domestic_remittances"].replace(
    {"Don't know/Refused": "Not Applicable"}
)

df["economy"] = df["economy"].astype(str)
df["age"] = df["age"].astype(float)
df["age_c"] = (
    df["age"] - df["age"].mean()
) / 10.0  # scaled to decades for numerical stability
df["age_c_sq"] = df["age_c"] ** 2

# Recode the varialbes to 0/1
for col in [
    "receive_wages",
    "receive_transfers",
    "receive_pensions",
    "receive_agriculture",
]:
    df[col + "_recv"] = (~df[col].isin(["Did not receive", "DK/ref"])).astype(int)

# Additional digital-finance and risk-management indicators
df["dig_account_bin"] = (df["dig_account"] == "Yes").astype(int)
df["merchantpay_dig_bin"] = (df["merchantpay_dig"] == "Yes").astype(int)
df["has_insurance_bin"] = (df["fin42"] == "Yes").astype(int)
df["has_mobile_phone_bin"] = (df["con1"] == "Yes").astype(int)


#     "could not afford medical care (fin28) / school fees in the past year (fin29)".
#     These are retained as explicit 3-level categoricals (Not
#     Applicable / No / Yes) because the skip pattern (only asked of
#     respondents who actually needed the service) is itself informative.
df["fin28_cat"] = df["fin28"].fillna("Not Applicable")
df["fin29_cat"] = df["fin29"].fillna("Not Applicable")

# Save cleaned analytic file
df.to_csv(OUT_PATH, index=False)
print(df.shape)
print("fin45_cat")
print(df["fin45_cat"].value_counts())
print("\nCountry distribution:\n", df["economy"].value_counts())

### EDA

In [ ]:
"""
01_eda.py
---------
Exploratory Data Analysis (EDA).
Produces: missingness table, frequency tables for the dependent variable
and every predictor, and publication-ready univariate bar charts.
"""


sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update(
    {
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "font.family": "DejaVu Sans",
        "axes.titleweight": "bold",
    }
)

FIG = Path("figures")
TAB = Path("tables")
FIG.mkdir(exist_ok=True)
TAB.mkdir(exist_ok=True)

df = pd.read_csv("data/saarc_clean.csv")

predictors = [
    "female",
    "educ",
    "inc_q",
    "emp_in",
    "urbanicity",
    "account_fin",
    "account_mob",
    "saved",
    "borrowed",
    "anydigpayment",
    "internet_use",
    "fin24",
    "fin24a",
    "pay_utilities",
    "domestic_remittances",
]
new_predictors = [
    "economy",
    "receive_wages",
    "receive_transfers",
    "receive_pensions",
    "receive_agriculture",
    "dig_account",
    "merchantpay_dig",
    "con1",
    "fin28",
    "fin29",
]

used_columns = ["fin45"] + predictors + new_predictors

# Missingness table (based on ORIGINAL raw values, before recoding)
raw = pd.read_csv("data/saarc_data_full.csv")
raw = raw[used_columns]
miss = raw.isna().sum().to_frame("n_missing")
miss["pct_missing"] = (miss["n_missing"] / len(raw) * 100).round(2)
miss = miss[miss["n_missing"] > 0].sort_values("n_missing", ascending=False)
miss.to_csv(TAB / "table_missingness.csv")
print("Missingness:\n", miss)

# Frequency table for the dependent variable
target_freq = df["fin45_cat"].value_counts().to_frame("n")
target_freq["pct"] = (target_freq["n"] / len(df) * 100).round(2)

# Figure 1: distribution of the dependent variable
fig, ax = plt.subplots(figsize=(9, 6))
order = target_freq.index
sns.barplot(
    x=target_freq["pct"], y=order, hue=order, palette="crest", legend=False, ax=ax
)
for i, v in enumerate(target_freq["pct"]):
    ax.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=12)
ax.set_xlabel("Share of respondents (%)")
ax.set_ylabel("")
ax.set_title("Figure 1. Distribution of Greatest Financial Worry (fin45)")
ax.set_xlim(0, target_freq["pct"].max() + 8)
plt.tight_layout()
plt.savefig(FIG / "fig01_target_distribution.png")
plt.close()

# Country and age distributions
country_freq = df["economy"].value_counts().to_frame("n")
country_freq["pct"] = (country_freq["n"] / len(df) * 100).round(2)
country_freq.to_csv(TAB / "table_freq_economy.csv")

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
sns.barplot(
    x=country_freq["pct"],
    y=country_freq.index,
    hue=country_freq.index,
    palette="crest",
    legend=False,
    ax=axes[0],
)
axes[0].set_title("Country (economy) composition of the pooled sample")
axes[0].set_xlabel("%")
sns.histplot(df["age"], bins=30, color="#4C72B0", ax=axes[1])
axes[1].set_title("Age distribution")
axes[1].set_xlabel("Age (years)")
fig.suptitle("Figure 2. Country Composition and Age Distribution", fontweight="bold")
plt.tight_layout()
plt.savefig(FIG / "fig02_country_age.png")
plt.close()

# Frequency tables + bar charts for every predictor
all_freqs = {}
for col in predictors:
    freq = df[col].value_counts().to_frame("n")
    freq["pct"] = (freq["n"] / len(df) * 100).round(2)
    all_freqs[col] = freq
    freq.to_csv(TAB / f"table_freq_{col}.csv")

# Composite Figure 3: grid of predictor distributions (demographic /
# access block) to keep the report concise while still publication ready
demo_cols = ["female", "educ", "inc_q", "emp_in", "urbanicity"]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(demo_cols):
    f = all_freqs[col].sort_values("pct", ascending=False)
    sns.barplot(
        x=f["pct"], y=f.index, hue=f.index, palette="mako", legend=False, ax=axes[i]
    )
    axes[i].set_title(col)
    axes[i].set_xlabel("%")
    axes[i].set_ylabel("")
axes[-1].axis("off")
fig.suptitle(
    "Figure 3. Distribution of Demographic and Labour-Market Predictors",
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(FIG / "fig03_demographic_distributions.png")
plt.close()

fin_cols = [
    "account_fin",
    "account_mob",
    "saved",
    "borrowed",
    "anydigpayment",
    "internet_use",
]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(fin_cols):
    f = all_freqs[col].sort_values("pct", ascending=False)
    sns.barplot(
        x=f["pct"], y=f.index, hue=f.index, palette="flare", legend=False, ax=axes[i]
    )
    axes[i].set_title(col)
    axes[i].set_xlabel("%")
    axes[i].set_ylabel("")
fig.suptitle(
    "Figure 4. Distribution of Financial Inclusion and Digital Finance Indicators",
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(FIG / "fig04_finance_distributions.png")
plt.close()

print("\nEDA complete. Figures written to figures/, tables written to tables/.")

### Bivariate

In [ ]:
"""
02_bivariate.py
----------------
Bivariate analysis of the dependent variable (fin45_cat) against every
categorical predictor: contingency tables, Pearson chi-square tests of
independence, and Cramer's V effect size. Also produces stacked
percentage-bar visualisations for the four predictors with the largest
association with financial worry.
"""


sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 300})

FIG = Path("figures")
TAB = Path("tables")

df = pd.read_csv("data/saarc_clean.csv")

predictors = [
    "female",
    "educ",
    "inc_q",
    "emp_in",
    "urbanicity",
    "account_fin",
    "account_mob",
    "saved",
    "borrowed",
    "anydigpayment",
    "internet_use",
    "fin24",
    "fin24a",
    "pay_utilities",
    "domestic_remittances",
    "economy",
    "receive_wages",
    "receive_transfers",
    "receive_pensions",
    "receive_agriculture",
    "dig_account",
    "merchantpay_dig",
    "con1",
    "fin28",
    "fin29",
]


def cramers_v(confusion_matrix: np.ndarray) -> float:
    """Bias-corrected Cramer's V (Bergsma, 2013)."""
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    return float(np.sqrt(phi2corr / min((kcorr - 1), (rcorr - 1))))


results = []
for col in predictors:
    ct = pd.crosstab(df[col], df["fin45_cat"])
    chi2, p, dof, _ = chi2_contingency(ct)
    v = cramers_v(ct.values)
    results.append(
        {
            "predictor": col,
            "chi2": round(chi2, 2),
            "dof": dof,
            "p_value": p,
            "cramers_v": round(v, 3),
        }
    )

assoc = pd.DataFrame(results).sort_values("cramers_v", ascending=False)
assoc["p_value_fmt"] = assoc["p_value"].apply(
    lambda p: "<0.001" if p < 0.001 else f"{p:.3f}"
)

# Age (continuous) vs. fin45_cat: Kruskal-Wallis H-test (chi-square/
#     Cramer's V requires categorical predictors, so age is tested
#     separately) plus group-wise mean age.

groups = [g["age"].values for _, g in df.groupby("fin45_cat")]
h_stat, p_kw = kruskal(*groups)
age_means = df.groupby("fin45_cat")["age"].agg(["mean", "std", "count"]).round(1)
age_means.to_csv(TAB / "table_age_by_worry.csv")

# Figure 5: Cramer's V ranking (effect-size lollipop chart)
fig, ax = plt.subplots(figsize=(10, 8))
order = assoc.sort_values("cramers_v")
ax.hlines(
    y=order["predictor"], xmin=0, xmax=order["cramers_v"], color="#4C72B0", linewidth=2
)
ax.plot(order["cramers_v"], order["predictor"], "o", color="#C44E52", markersize=9)
ax.set_xlabel("Cramer's V (association with fin45_cat)")
ax.set_title(
    "Figure 5. Bivariate Association of Predictors with\nGreatest Financial Worry (Cramer's V)"
)
plt.tight_layout()
plt.savefig(FIG / "fig05_cramers_v_ranking.png")
plt.close()

# Figure 6: stacked 100% bar chart, top 4 predictors by Cramer's V
top4 = assoc["predictor"].head(4).tolist()
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()
for i, col in enumerate(top4):
    ct = pd.crosstab(df[col], df["fin45_cat"], normalize="index") * 100
    ct = ct[df["fin45_cat"].value_counts().index]  # consistent class order
    ct.plot(kind="bar", stacked=True, ax=axes[i], colormap="viridis", legend=False)
    axes[i].set_title(
        f"{col}  (Cramer's V = {assoc.set_index('predictor').loc[col, 'cramers_v']:.3f})"
    )
    axes[i].set_ylabel("% within category")
    axes[i].set_xlabel("")
    axes[i].tick_params(axis="x", rotation=30)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.05))
fig.suptitle(
    "Figure 6. Composition of Greatest Financial Worry by Top-4 Associated Predictors",
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(FIG / "fig06_stacked_top4_predictors.png", bbox_inches="tight")
plt.close()

print("\nBivariate analysis complete.")

### Feature Engineering

In [ ]:
"""
03_feature_engineering.py
--------------------------
Encodes predictors for modelling:
  - Ordinal encoding for genuinely ordinal variables (educ, inc_q, fin24a)
  - One-hot (dummy) encoding for nominal variables, including country identifier (economy) and two financial-stress
    history variables (fin28_cat, fin29_cat)
  - Binary (0/1) recoding for Yes/No-type indicators, including four 
    income-source variables and three digital-finance/insurance
    indicators
  - Continuous age (mean-centred) plus a quadratic term to capture a
    possible non-linear (hump/U-shaped) life-cycle relationship with
    financial worry
Builds a stratified 70/30 train-test split and persists the design
matrices so that every model in scripts 04-07 is trained/evaluated on an
identical split (fair comparison across the multinomial logit, SVM,
Random Forest and XGBoost).
"""

df = pd.read_csv("data/saarc_clean.csv")

# Ordinal encodings
educ_order = {
    "Primary education or less": 0,
    "Secondary education": 1,
    "Tertiary education or more": 2,
}
incq_order = {"Poorest": 0, "Poor": 1, "Middle": 2, "Rich": 3, "Richest": 4}
fin24a_order = {
    "Not Applicable": -1,
    "Not difficult at all": 0,
    "Somewhat difficult": 1,
    "Very difficult": 2,
    "Don't know/Refused": 1,
}  # DK imputed at the sample median category

df["educ_ord"] = df["educ"].map(educ_order)
df["inc_q_ord"] = df["inc_q"].map(incq_order)
df["fin24a_ord"] = df["fin24a"].map(fin24a_order)

# Binary recodes (Yes = 1)
binary_cols = [
    "account_fin",
    "account_mob",
    "saved",
    "borrowed",
    "anydigpayment",
    "internet_use",
]
for col in binary_cols:
    df[col + "_bin"] = (df[col] == "Yes").astype(int)

df["female_bin"] = (df["female"] == "Female").astype(int)
df["urban_bin"] = (df["urbanicity"] == "Urban").astype(int)
df["employed_bin"] = (df["emp_in"] == "In the workforce").astype(int)

# Nominal one-hot encodings (drop_first to avoid the dummy trap)
nominal_cols = ["fin24", "pay_utilities", "domestic_remittances"]
dummies = pd.get_dummies(df[nominal_cols], prefix=nominal_cols, drop_first=True)

feature_cols_ordinal_binary = [
    "female_bin",
    "educ_ord",
    "inc_q_ord",
    "employed_bin",
    "urban_bin",
    "account_fin_bin",
    "account_mob_bin",
    "saved_bin",
    "borrowed_bin",
    "anydigpayment_bin",
    "internet_use_bin",
    "fin24a_ord",
]

# Additional features enabled by the full 200-column extract
new_binary_cols = [
    "receive_wages_recv",
    "receive_transfers_recv",
    "receive_pensions_recv",
    "receive_agriculture_recv",
    "dig_account_bin",
    "merchantpay_dig_bin",
    "has_mobile_phone_bin",
]
# Note: has_insurance_bin (fin42) was found to be perfectly collinear
# (r = 1.00, identical values) with receive_agriculture_recv in this
# extract and is therefore dropped to avoid infinite VIF; only one of
# the pair is retained.
feature_cols_ordinal_binary += new_binary_cols
feature_cols_ordinal_binary += ["age_c", "age_c_sq"]

# fin28/fin29 simplified to a single binary flag (Yes vs. No/Not-Applicable)
df["fin28_bin"] = (df["fin28_cat"] == "Yes").astype(int)
df["fin29_bin"] = (df["fin29_cat"] == "Yes").astype(int)
feature_cols_ordinal_binary += ["fin28_bin", "fin29_bin"]

new_nominal_cols = ["economy"]
new_dummies = pd.get_dummies(
    df[new_nominal_cols], prefix=new_nominal_cols, drop_first=True
)
dummies = pd.concat([dummies, new_dummies], axis=1)

X = pd.concat([df[feature_cols_ordinal_binary], dummies], axis=1)
X.columns = [c.replace(" ", "_").replace(",", "").replace("/", "_") for c in X.columns]
y = df["fin45_cat"]

print("Feature matrix shape:", X.shape)
print("Features:", X.columns.tolist())

# Stratified train/test split (70/30), fixed random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

Path("data").mkdir(exist_ok=True)
X_train.to_csv("data/X_train.csv", index=False)
X_test.to_csv("data/X_test.csv", index=False)
y_train.to_csv("data/y_train.csv", index=False)
y_test.to_csv("data/y_test.csv", index=False)
joblib.dump(list(X.columns), "data/feature_names.pkl")

print("\nTrain shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train class balance:\n", y_train.value_counts(normalize=True).round(3))

### Diagnostics

In [ ]:
"""
04_diagnostics.py
------------------
Diagnostic checks prior to multinomial logit estimation:
  - Pearson correlation matrix of the (numeric-encoded) design matrix
  - Variance Inflation Factors (VIF) for every predictor
A VIF > 10 (or, more conservatively, > 5) signals problematic
multicollinearity; results are reported in Chapter 4.
"""


sns.set_theme(style="white", context="talk")
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 300})

FIG = Path("figures")
TAB = Path("tables")

X_train = pd.read_csv("data/X_train.csv").astype(float)

# Correlation heatmap
corr = X_train.corr()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    corr,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.3,
    cbar_kws={"shrink": 0.7},
    ax=ax,
)
ax.set_title("Figure 7. Correlation Matrix of Encoded Predictors", fontweight="bold")
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(FIG / "fig07_correlation_heatmap.png")
plt.close()

# Variance Inflation Factors
X_const = add_constant(X_train)
vif = pd.DataFrame(
    {
        "feature": X_const.columns,
        "VIF": [
            variance_inflation_factor(X_const.values, i)
            for i in range(X_const.shape[1])
        ],
    }
)
vif = vif[vif["feature"] != "const"].sort_values("VIF", ascending=False)
vif.to_csv(TAB / "table_vif.csv", index=False)
print(vif.to_string(index=False))

max_vif = vif["VIF"].max()
print(
    f"\nMax VIF = {max_vif:.2f}  ->",
    "no serious multicollinearity (all VIF < 5)"
    if max_vif < 5
    else "moderate/serious multicollinearity detected",
)

### Multinomial Login

In [ ]:
"""
05_multinomial_logit.py
-------------------------
Estimates a multinomial logistic regression (MNLogit) of the greatest
financial worry (fin45_cat) on the demographic, labour-market and
financial-inclusion predictor set. "Daily/Monthly Expenses" (the modal
category) is the reference outcome. Reports coefficients, relative risk
ratios (RRR = exp(beta)), McFadden's pseudo-R2, and an in-sample /
out-of-sample classification performance summary so the parametric model
can be benchmarked against the machine-learning classifiers in script 06.
"""


TAB = Path("tables")

X_train = pd.read_csv("data/X_train.csv").astype(float)
X_test = pd.read_csv("data/X_test.csv").astype(float)
y_train = pd.read_csv("data/y_train.csv").squeeze("columns")
y_test = pd.read_csv("data/y_test.csv").squeeze("columns")

# Reference category = modal class
REFERENCE = "Daily/Monthly Expenses"
categories = [REFERENCE] + [c for c in y_train.unique() if c != REFERENCE]
cat_dtype = pd.CategoricalDtype(categories=categories, ordered=False)
y_train_cat = y_train.astype(cat_dtype).cat.codes
y_test_cat = y_test.astype(cat_dtype).cat.codes

X_train_c = sm.add_constant(X_train)
X_test_c = sm.add_constant(X_test, has_constant="add")

model = sm.MNLogit(y_train_cat, X_train_c)
result = model.fit(method="bfgs", maxiter=2000, gtol=1e-5, disp=True)
print(result.summary())

# McFadden's pseudo-R2
llf = result.llf
llnull = result.llnull
pseudo_r2 = 1 - llf / llnull
print(f"\nMcFadden's pseudo-R2 = {pseudo_r2:.4f}")

# Relative risk ratios (RRR) table, one block per non-reference outcome
params = result.params
pvalues = result.pvalues
rrr = np.exp(params)
outcome_labels = [c for c in categories if c != REFERENCE]

rrr_tables = {}
for j, label in enumerate(outcome_labels):
    tbl = pd.DataFrame(
        {
            "coef": params.iloc[:, j],
            "RRR": rrr.iloc[:, j],
            "p_value": pvalues.iloc[:, j],
        }
    )
    tbl["sig"] = tbl["p_value"].apply(
        lambda p: "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""
    )
    rrr_tables[label] = tbl
    tbl.to_csv(
        TAB / f"table_mnlogit_RRR_{label.replace('/', '_').replace(' ', '_')}.csv"
    )

print(
    "\nRRR table (vs. reference = 'Daily/Monthly Expenses') written for outcomes:",
    outcome_labels,
)


# Predictive performance (train and test)
def predict_labels(Xc):
    probs = result.predict(Xc)
    codes = np.argmax(probs.values, axis=1)
    return pd.Categorical.from_codes(codes, categories=categories)


pred_train = predict_labels(X_train_c)
pred_test = predict_labels(X_test_c)

acc_train = accuracy_score(y_train, pred_train)
acc_test = accuracy_score(y_test, pred_test)
f1_test_macro = f1_score(y_test, pred_test, average="macro")
f1_test_weighted = f1_score(y_test, pred_test, average="weighted")

print(f"\nTrain accuracy = {acc_train:.4f}")
print(f"Test accuracy  = {acc_test:.4f}")
print(f"Test macro-F1  = {f1_test_macro:.4f}")
print(f"Test weighted-F1 = {f1_test_weighted:.4f}")

report = classification_report(y_test, pred_test, digits=3)
print("\n", report)

# Macro-average one-vs-rest ROC-AUC (uses fitted probabilities), for a
# like-for-like comparison against the SVM / RF / XGBoost models in 06
proba_test = result.predict(X_test_c).values
try:
    auc_test = roc_auc_score(y_test_cat, proba_test, multi_class="ovr", average="macro")
except Exception:
    auc_test = np.nan
print(f"Test macro-AUC (OvR) = {auc_test:.4f}")

# Save model comparison row for later aggregation
pd.DataFrame(
    [
        {
            "model": "Multinomial Logistic Regression",
            "accuracy": acc_test,
            "f1_macro": f1_test_macro,
            "f1_weighted": f1_test_weighted,
            "roc_auc_macro": auc_test,
        }
    ]
).to_csv(TAB / "modelcomp_mnlogit.csv", index=False)

pd.DataFrame(
    [
        {
            "pseudo_r2_mcfadden": pseudo_r2,
            "llf": llf,
            "llnull": llnull,
            "n_obs": int(result.nobs),
        }
    ]
).to_csv(TAB / "table_mnlogit_fit.csv", index=False)

### Models Comparison

In [ ]:
"""
06_ml_comparison.py
---------------------
Trains and evaluates three machine-learning classifiers on the identical
train/test split used for the multinomial logit (script 05):
  1. Support Vector Machine (RBF kernel, probability=True)
  2. Random Forest
  3. XGBoost (multi:softprob)
Reports accuracy, macro-F1, weighted-F1, and multiclass ROC-AUC (OvR),
saves confusion matrices, and writes a consolidated model-comparison
table + bar chart that also incorporates the multinomial logit results
from script 05.
"""


warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 300})

FIG = Path("figures")
TAB = Path("tables")

X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")
y_train = pd.read_csv("data/y_train.csv").squeeze("columns")
y_test = pd.read_csv("data/y_test.csv").squeeze("columns")

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)
class_names = le.classes_
joblib.dump(le, "data/label_encoder.pkl")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

results = []
fitted_models = {}

# Support Vector Machine (RBF kernel)
svm = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    class_weight="balanced",
    random_state=42,
)
svm.fit(X_train_sc, y_train_enc)
pred_svm = svm.predict(X_test_sc)
proba_svm = svm.predict_proba(X_test_sc)
fitted_models["SVM"] = svm

# Random Forest
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train_enc)
pred_rf = rf.predict(X_test)
proba_rf = rf.predict_proba(X_test)
fitted_models["Random Forest"] = rf

# XGBoost
sample_weight = (
    pd.Series(y_train_enc)
    .map((1 / pd.Series(y_train_enc).value_counts(normalize=True)))
    .values
)

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=len(class_names),
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)
xgb.fit(X_train, y_train_enc, sample_weight=sample_weight)
pred_xgb = xgb.predict(X_test)
proba_xgb = xgb.predict_proba(X_test)
fitted_models["XGBoost"] = xgb

joblib.dump(fitted_models, "data/fitted_ml_models.pkl")
joblib.dump(scaler, "data/scaler.pkl")


# Evaluation helper
def evaluate(name, y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro")
    f1w = f1_score(y_true, y_pred, average="weighted")
    try:
        auc = roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro")
    except Exception:
        auc = np.nan
    print(f"\n=== {name} ===")
    print(
        f"Accuracy={acc:.4f}  Macro-F1={f1m:.4f}  Weighted-F1={f1w:.4f}  Macro-AUC={auc:.4f}"
    )
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
    results.append(
        {
            "model": name,
            "accuracy": acc,
            "f1_macro": f1m,
            "f1_weighted": f1w,
            "roc_auc_macro": auc,
        }
    )
    return confusion_matrix(y_true, y_pred)


cm_svm = evaluate("Support Vector Machine (RBF)", y_test_enc, pred_svm, proba_svm)
cm_rf = evaluate("Random Forest", y_test_enc, pred_rf, proba_rf)
cm_xgb = evaluate("XGBoost", y_test_enc, pred_xgb, proba_xgb)

# # Confusion matrices figure
sns.set_theme(style="white")
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
models = [
    ("SVM (RBF)", cm_svm),
    ("Random Forest", cm_rf),
    ("XGBoost", cm_xgb),
]

for ax, (title, cm) in zip(axes, models):
    # Row-normalized percentages
    cm_pct = cm.astype(float)
    cm_pct = cm_pct / cm_pct.sum(axis=1, keepdims=True) * 100

    sns.heatmap(
        cm_pct,
        annot=True,
        fmt=".1f",
        cmap="Blues",
        linewidths=0.8,
        linecolor="white",
        square=True,
        cbar=False,
        xticklabels=class_names,
        yticklabels=class_names,
        annot_kws={"fontsize": 11, "fontweight": "bold"},
        ax=ax,
    )

    ax.set_title(title, fontsize=15, fontweight="bold", pad=12)
    ax.set_xlabel(
        "Predicted Label",
        fontsize=12,
        fontweight="bold",
        labelpad=10,
    )
    ax.set_ylabel(
        "True Label",
        fontsize=12,
        fontweight="bold",
        labelpad=10,
    )
    ax.tick_params(axis="x", labelrotation=30, labelsize=10)
    ax.tick_params(axis="y", labelrotation=0, labelsize=10)

plt.suptitle(
    "Figure 8. Row-normalized Confusion Matrices (%) on Test Set",
    fontsize=18,
    fontweight="bold",
    y=1.02,
)
plt.savefig(
    FIG / "fig08_confusion_matrices.png",
    dpi=300,
    bbox_inches="tight",
)

# Consolidated model-comparison table + chart (adds mnlogit from script 05)
ml_results = pd.DataFrame(results)
mnlogit_row = pd.read_csv(TAB / "modelcomp_mnlogit.csv")
comparison = pd.concat([mnlogit_row, ml_results], ignore_index=True)
comparison.to_csv(TAB / "table_model_comparison.csv", index=False)
print("\nModel comparison:\n", comparison)

# Random Forest feature importance (native) for cross-check against SHAP
rf_imp = pd.DataFrame(
    {"feature": X_train.columns, "importance": rf.feature_importances_}
)
rf_imp = rf_imp.sort_values("importance", ascending=False)
rf_imp.to_csv(TAB / "table_rf_feature_importance.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 9))
top15 = rf_imp.head(15).sort_values("importance")
ax.barh(top15["feature"], top15["importance"], color="#55A868")
ax.set_title("Figure 9. Random Forest Feature Importance (Top 15)")
ax.set_xlabel("Mean Decrease in Impurity")
plt.tight_layout()
plt.savefig(FIG / "fig09_rf_feature_importance.png")

print("\nML comparison complete.")

### SHAP Analysis

In [ ]:
"""
07_shap_analysis.py
---------------------
SHAP (SHapley Additive exPlanations) analysis of the best tree-based
classifier (XGBoost) to interpret feature contributions to each
financial-worry category. Produces a global mean(|SHAP|) importance
ranking (bar) and a class-specific beeswarm summary plot for the modal
outcome, "Daily/Monthly Expenses", plus the top competing outcome,
"Medical Emergency".
"""


plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 300})

FIG = Path("figures")
TAB = Path("tables")

X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")
le = joblib.load("data/label_encoder.pkl")
models = joblib.load("data/fitted_ml_models.pkl")
xgb = models["XGBoost"]
class_names = le.classes_

# Use a manageable background/explain sample for speed & readability
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

# Normalise shape across shap versions: want a list of (n_samples, n_features) per class
if isinstance(shap_values, list):
    sv_list = shap_values
else:
    sv_arr = np.array(shap_values)
    if sv_arr.ndim == 3 and sv_arr.shape[-1] == len(class_names):
        sv_list = [sv_arr[:, :, k] for k in range(len(class_names))]
    else:
        sv_list = [sv_arr[k] for k in range(sv_arr.shape[0])]

# Global feature importance: mean(|SHAP|) averaged across all classes
mean_abs_per_class = np.stack(
    [np.abs(sv).mean(axis=0) for sv in sv_list]
)  # (n_classes, n_features)
global_importance = mean_abs_per_class.mean(axis=0)
imp_df = pd.DataFrame({"feature": X_test.columns, "mean_abs_shap": global_importance})
imp_df = imp_df.sort_values("mean_abs_shap", ascending=False)
imp_df.to_csv(TAB / "table_shap_global_importance.csv", index=False)
print(imp_df.head(15).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 9))
top15 = imp_df.head(15).sort_values("mean_abs_shap")
ax.barh(top15["feature"], top15["mean_abs_shap"], color="#C44E52")
ax.set_title(
    "Figure 10. Global SHAP Feature Importance — XGBoost\n(mean |SHAP value|, averaged across all outcome classes)"
)
ax.set_xlabel("Mean |SHAP value|")
plt.tight_layout()
plt.savefig(FIG / "fig10_shap_global_importance.png")
plt.close()

# Class-specific beeswarm summary plots
focus_classes = ["Daily/Monthly Expenses", "Medical Emergency", "Education", "Business"]

imgs = []
for cls in focus_classes:
    idx = list(class_names).index(cls)
    plt.figure(figsize=(10, 9))
    shap.summary_plot(sv_list[idx], X_test, show=False, plot_size=(10, 9))
    plt.title(f"SHAP Summary — Outcome: {cls}", fontsize=14, fontweight="bold")
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=150)
    plt.close()
    buf.seek(0)
    imgs.append(Image.open(buf))

fig, axes = plt.subplots(2, 2, figsize=(20, 18))
for ax, img, cls in zip(axes.flatten(), imgs, focus_classes):
    ax.imshow(img)
    ax.axis("off")

plt.tight_layout()
combined_path = FIG / "fig_shap_summary_combined.png"
plt.savefig(combined_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved combined SHAP summary plot: {combined_path}")
print("\nSHAP analysis complete.")